In [ ]:
import sys
import os
import io
import contextlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'parameters'))
sys.path.insert(0, os.getcwd())

from analysis_utils import (
    load_best_runs, load_all_runs, extract_history, add_wall_times,
    build_summary_dataframe,
    plot_convergence_curves, plot_speedrun_results,
    plot_convergence_curves_grid, plot_multi_function_performance,
    plot_best_of_budget, plot_best_of_budget_grid,
    print_ranking_table, print_variant_comparison,
    print_variant_comparison_allruns, print_offdiag_ranking,
    bootstrap_best_run_ci, print_topk_comparison, print_robustness_table,
    print_best_hyperparameters, save_best_hyperparameters,
    print_hp_sensitivity_table,
    get_optimizer_colors, ema,
    TASK_CONFIGS,
)

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

BACKEND = "local"
RESULTS_DIR = os.path.join('..', '..', 'results')


def _quiet(fn, *args, **kwargs):
    """Call fn while suppressing stdout (loader noise)."""
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kwargs)

In [ ]:
def load_task(task_key):
    """Load all data for a sweep task."""
    cfg = TASK_CONFIGS[task_key]
    colors = get_optimizer_colors(cfg['optimizers'])

    best_runs = _quiet(load_best_runs,
        backend=BACKEND, optimizers=cfg['optimizers'],
        task_tag=cfg['task_tag'], results_dir=RESULTS_DIR,
        metric_key=cfg['metric_key'], direction=cfg['direction'],
        sort_metric=cfg['sort_metric'], sort_order=cfg['sort_order'],
        iteration=cfg['iteration'],
    )
    all_runs = _quiet(load_all_runs,
        backend=BACKEND, optimizers=cfg['optimizers'],
        task_tag=cfg['task_tag'], results_dir=RESULTS_DIR,
        iteration=cfg['iteration'],
    )
    raw_data = _quiet(extract_history, BACKEND, best_runs, cfg['history_metrics'])
    add_wall_times(raw_data)

    smoothed = {}
    for opt, data in raw_data.items():
        d = dict(data)
        for key, fn in cfg.get('y_transforms', {}).items():
            d[key] = fn(d)
        for key in cfg['y_keys']:
            if key in d and len(d[key]) > 0:
                d[key] = ema(np.asarray(d[key], dtype=float))
        smoothed[opt] = d

    print(f"Loaded {len(best_runs)} optimizers, {sum(len(v) for v in all_runs.values())} total runs")
    return cfg, best_runs, all_runs, smoothed, raw_data, colors


def show_results(cfg, best_runs, all_runs, smoothed, raw_data, colors):
    """Plot all results for a single sweep task."""
    name = cfg['display_name']
    itr = cfg['iteration']

    fig = plot_convergence_curves(
        smoothed, y_keys=cfg['y_keys'], x_keys=['epoch', 'wall_times'],
        mark_best=False, colors=colors, highlight=cfg.get('highlight'),
        y_labels=cfg.get('y_labels'),
        x_labels={'epoch': 'Epoch', 'wall_times': 'Wall Time (s)'},
        log_y=cfg.get('log_y', 'auto'),
    )
    for ax in fig.axes:
        for yk, lims in cfg.get('ylim', {}).items():
            if ax.get_ylabel() == cfg.get('y_labels', {}).get(yk, yk):
                ax.set_ylim(*lims)
    plt.tight_layout()
    plt.show()

    fig = plot_speedrun_results(
        raw_data, cfg['speedrun_targets'],
        metric_key=cfg['speedrun_metric'],
        direction=cfg['speedrun_direction'], colors=colors,
    )
    plt.show()

    print_ranking_table(best_runs, metric_key=cfg['metric_key'],
                        direction=cfg['direction'],
                        title=f'{name} (itr {itr}) — Optimizer Ranking')
    print_variant_comparison(best_runs, metric_key=cfg['metric_key'],
                             direction=cfg['direction'],
                             title=f'{name} (itr {itr}) — Variant Pairs (Best Run)')
    print_variant_comparison_allruns(all_runs, metric_key=cfg['metric_key'],
                                     direction=cfg['direction'],
                                     title=f'{name} (itr {itr}) — Variant Comparison (All Runs)')
    print_offdiag_ranking(best_runs, metric_key=cfg['metric_key'],
                          direction=cfg['direction'],
                          title=f'{name} (itr {itr}) — Off-Diagonal (a,b) Ranking')
    bootstrap_best_run_ci(all_runs, metric_key=cfg['metric_key'],
                          direction=cfg['direction'],
                          title=f'{name} (itr {itr}) — Bootstrap 95% CI')
    print_topk_comparison(all_runs, metric_key=cfg['metric_key'],
                          direction=cfg['direction'],
                          title=f'{name} (itr {itr}) — Top-k Mean Comparison')
    print_robustness_table(all_runs, metric_key=cfg['metric_key'],
                           direction=cfg['direction'],
                           title=f'{name} (itr {itr}) — Robustness Metrics')

    fig = plot_best_of_budget(
        all_runs, metric_key=cfg['metric_key'],
        direction=cfg['direction'], colors=colors,
        title=f'{name} (itr {itr}) — Best-of-Budget',
    )
    plt.show()

    print_best_hyperparameters(best_runs)
    filename = f"best_hyperparameters_{cfg['task_tag']}_itr_{itr}.csv"
    save_best_hyperparameters(best_runs, filename)

# MNIST MLP

In [ ]:
mnist = load_task("mnist_mlp")
show_results(*mnist)

# CIFAR-10 ResNet-18

In [ ]:
cifar = load_task("cifar10_resnet18")
show_results(*cifar)

# Shakespeare MiniGPT

In [ ]:
shake = load_task("shakespeare_minigpt")
show_results(*shake)

# Regression

In [ ]:
reg = load_task("regression")
show_results(*reg)

# Small Examples (Beale, Rosenbrock, Himmelblau, Ackley, Rastrigin, Styblinski-Tang)

In [ ]:
FUNCTIONS = ['beale', 'rosenbrock', 'himmelblau', 'ackley', 'rastrigin', 'styblinski_tang']
ITERATION_SE = 4
HIGHLIGHT = ["sgd_learn_diag_curv", "sgd_learn_diag_curv_log"]

# Auto-detect optimizers
if BACKEND == "local":
    _avail = set()
    for fn in FUNCTIONS:
        td = os.path.join(RESULTS_DIR, f"small_examples_{fn}")
        if os.path.isdir(td):
            _avail.update(d for d in os.listdir(td) if os.path.isdir(os.path.join(td, d)))
    SE_OPTIMIZERS = sorted(_avail)
else:
    SE_OPTIMIZERS = ['adam', 'sgd_learn_diag', 'sgd_learn_diag_curv']

se_colors = get_optimizer_colors(SE_OPTIMIZERS)

# Load best runs per function
all_best_runs = {}
for fn in FUNCTIONS:
    all_best_runs[fn] = _quiet(load_best_runs,
        backend=BACKEND, optimizers=SE_OPTIMIZERS,
        task_tag=f"small_examples_{fn}", results_dir=RESULTS_DIR,
        metric_key="sweep_metric", direction="minimize",
        sort_metric="sweep_metric", sort_order="+", iteration=ITERATION_SE,
    )

# Extract history
all_history_data = {}
for fn in FUNCTIONS:
    br = all_best_runs[fn]
    if br:
        hist = _quiet(extract_history, BACKEND, br, ["function_value", "runtime_seconds"])
        add_wall_times(hist, time_key="runtime_seconds")
        all_history_data[fn] = hist

# Load all runs per function
all_runs_by_func = {}
for fn in FUNCTIONS:
    all_runs_by_func[fn] = _quiet(load_all_runs,
        backend=BACKEND, optimizers=SE_OPTIMIZERS,
        task_tag=f"small_examples_{fn}", results_dir=RESULTS_DIR,
        iteration=ITERATION_SE,
    )

summary_df = build_summary_dataframe(all_best_runs, optimizers=SE_OPTIMIZERS)
print(f"Loaded {len(FUNCTIONS)} functions, {len(SE_OPTIMIZERS)} optimizers, {len(summary_df)} summary rows")

# --- Performance summary (6-panel dashboard) ---
if len(summary_df) > 0:
    fig = plot_multi_function_performance(summary_df)
    plt.show()

# --- Convergence curves grid ---
fig = plot_convergence_curves_grid(
    all_history_data, FUNCTIONS,
    highlight=HIGHLIGHT, colors=se_colors,
    best_runs_by_func=all_best_runs,
)
plt.show()

# --- Rankings (pooled across functions) ---
pooled_runs = {}
for fn in FUNCTIONS:
    for opt, runs in all_runs_by_func.get(fn, {}).items():
        pooled_runs.setdefault(opt, []).extend(runs)

print_variant_comparison_allruns(pooled_runs, metric_key="sweep_metric", direction="minimize",
                                 title='Small Examples (pooled) — Variant Comparison (Mann-Whitney U)')

bootstrap_best_run_ci(pooled_runs, metric_key="sweep_metric", direction="minimize",
                      title=f'Small Examples (pooled, itr {ITERATION_SE}) — Bootstrap 95% CI')
print_topk_comparison(pooled_runs, metric_key="sweep_metric", direction="minimize",
                      title=f'Small Examples (pooled, itr {ITERATION_SE}) — Top-k Mean Comparison')
print_robustness_table(pooled_runs, metric_key="sweep_metric", direction="minimize",
                       title=f'Small Examples (pooled, itr {ITERATION_SE}) — Robustness Metrics')

# --- Best-of-budget grid ---
fig = plot_best_of_budget_grid(
    all_runs_by_func, FUNCTIONS,
    metric_key="sweep_metric", direction="minimize",
    colors=se_colors,
)
plt.show()

# --- HP export ---
for fn in FUNCTIONS:
    br = all_best_runs[fn]
    if br:
        print_best_hyperparameters(br)
        save_best_hyperparameters(br, f"best_hyperparameters_small_examples_{fn}_itr_{ITERATION_SE}.csv")